# NLP project: API model runs (GPT-6 Luna, Gemma 4 26B-A4B, Gemini 3.5 Flash-Lite) and scoring

All code is in this notebook: prompt building, answer parsing, API clients, the resumable runner and the scorers.
They are copied verbatim from `scripts/` in the replication package. Only **data** is loaded from a folder (`DATA_DIR`), which contains
`share/nlp_project_sample366.jsonl`, `data/` (subset items, sample ids, Urdu options), `results/` (our raw outputs for all six models: the three API models and the three small models) and `expected_results.json`.
Get the data in one of three ways:
**(a)** in Colab, upload `nlp_project_data.zip` when asked;
**(b)** unzip it on Google Drive and set `DATA_DIR`, e.g. `/content/drive/MyDrive/nlp_project_data`;
**(c)** set `DATA_DIR` to a local folder, or run from the replication package root. Both are found automatically.

**Modes:**
- `'score_only'` (default): no API calls and no keys. Regenerates every table for **all six models** (3 API + 3 small) from the included
  raw outputs and asserts that the numbers match `expected_results.json`.
- `'run'`: re-runs the models with exactly our settings into `<DATA_DIR>/results_rerun/`. The included results are never touched.
  Resumable: re-running skips items already done. `SMOKE_ITEMS = 2` makes a tiny paid test first (`smoke_results/`).

**Settings (all runs):** sample file `share/nlp_project_sample366.jsonl`, original (released) Urdu questions, scripts en/ru/ur,
interleaved, shuffled with seed 42, temperature 0.1, max 2,048 output tokens, conditions `zs` (U-MIRAGE zero-shot) and `umcot` (U-MIRAGE "CoT" prompt).

| model | provider | thinking / reasoning | rpm / workers | price flags (USD per 1M in/out) | cost cap |
|---|---|---|---|---|---|
| `gpt-6-luna` | OpenAI Chat Completions | `reasoning_effort: none` | 900 / 16 | 0.10 / 0.50 | $3 |
| `gemma-4-26b-a4b-it` | Gemini API | `thinkingLevel: minimal` | 28 / 6, timeout 120 s | 0.15 / 0.60 | $2 |
| `gemini-3.5-flash-lite` | Gemini API | `thinkingLevel: minimal` | 60 / 4 | 0.30 / 2.50 | $2 |

**Keys** (only for `run`): `OPENAI_API_KEY`, `GEMINI_API_KEY` from Colab secrets or environment variables. They are never printed or saved.

**Step 1: Choose what to do.** Sets the mode. `score_only` (default) makes no API calls and only re-checks our reported numbers; `run` calls the models again. Leave `DATA_DIR` empty to find or upload the data automatically.

*Input:* nothing. *Output:* `MODE`, `RUN_MODELS`, `SMOKE_ITEMS`, `RERUN_DIR`, `DATA_DIR`.

In [ ]:
# ---- settings ----
MODE = 'score_only'       # 'score_only' (no API calls) or 'run'
RUN_MODELS = ['gpt-6-luna', 'gemma-4-26b-a4b-it', 'gemini-3.5-flash-lite']   # used when MODE == 'run'
SMOKE_ITEMS = 0           # >0: only the first N items per script/cond, written to smoke_results/
RERUN_DIR = 'results_rerun'
DATA_DIR = ''             # '' = auto-detect / upload; or e.g. '/content/drive/MyDrive/nlp_project_data'

## Data folder

**Step 2: Find the data.** Looks for the data folder (it must contain `share/nlp_project_sample366.jsonl`): the package root, a Drive folder, or a zip you upload in Colab. All later steps read from this folder, so the notebook itself holds only code.

*Input:* `DATA_DIR` or the uploaded `nlp_project_data.zip`. *Output:* `DATA_DIR` / `ROOT` pointing at the data folder.

In [ ]:
# Find the data folder: it must contain share/nlp_project_sample366.jsonl (plus data/, and results/ for score-only).
import os, sys, zipfile
IN_COLAB = 'google.colab' in sys.modules

def is_data_dir(d):
    return bool(d) and os.path.isfile(os.path.join(d, 'share', 'nlp_project_sample366.jsonl'))

if IN_COLAB and DATA_DIR.startswith('/content/drive'):          # (b) folder on Google Drive
    from google.colab import drive
    drive.mount('/content/drive')
here = os.getcwd()
candidates = [DATA_DIR, here, os.path.join(here, 'nlp_project_data'),          # (c) local path / package root
              os.path.dirname(here), os.path.join(os.path.dirname(here), 'nlp_project_data'),
              '/content/nlp_project_data']                                     # (a) uploaded earlier this session
found = next((os.path.abspath(d) for d in candidates if is_data_dir(d)), None)
if found is None and IN_COLAB:                                    # (a) upload the data zip now
    from google.colab import files
    print('Upload nlp_project_data.zip (or nlp_project_replication_package.zip)')
    uploaded = files.upload()
    zname = next(n for n in uploaded if n.endswith('.zip'))
    zipfile.ZipFile(zname).extractall('/content')
    found = next((d for d in ['/content/nlp_project_data', '/content/nlp_project_replication_package'] if is_data_dir(d)), None)
assert is_data_dir(found), 'Data folder not found: set DATA_DIR to the folder that contains share/, data/ (and results/).'
DATA_DIR = found
ROOT = DATA_DIR          # the code below (copied from scripts/) resolves all data paths relative to ROOT
print('data folder:', DATA_DIR)

**Step 3: Install the few libraries needed.** Installs the pinned versions of `requests`, `numpy` and `scipy` used for the API calls and the statistics.

*Input:* internet access. *Output:* installed packages.

In [ ]:
%pip install -q requests==2.34.2 numpy==2.5.5 scipy==1.18.1

## Code
#### 1. Prompts (U-MIRAGE Figure 1 wording) and the answer parser

**Step 4: Define the prompts and the answer parser.** Defines U-MIRAGE's zero-shot and "CoT" prompts (Figure 1 wording, verbatim) and the parser that pulls the answer letter out of a reply. Every model uses this same code, so the comparisons are fair.

*Input:* nothing. *Output:* the functions `build_prompt` and `parse_answer`.

In [ ]:
# --- from scripts/run_eval.py (verbatim; the CLI scripts use the same code) ---
import argparse, glob, json, os, random, re, sys, time, datetime as dt


import requests


LANG = {'en': 'English', 'ru': 'Roman Urdu', 'ur': 'Urdu'}


# U-MIRAGE Figure 1, verbatim wording.
ZS = ("This is a medical multiple-choice question written in {language}\n"
      "Question: {question}\nOptions: {options}\n"
      "Select the single correct option (A, B, C, D). Only return the letter corresponding to the correct answer. No explanation.")


# U-MIRAGE's own "CoT" prompt (kept for reference; it forbids written reasoning). Use --conds umcot to run it.
UMCOT = ("This is a medical multiple-choice question written in {language}\n"
         "Question: {question}\nOptions: {options}\n"
         "Let's think step-by-step to reason about the correct answer based on medical knowledge. After reasoning, "
         "select the single best option (A, B, C, or D). Return only the final answer letter (A, B, C, or D) without any explanation.")


# Our written-CoT condition: same frame, reasoning is written out, answer on the last line.
COT = ("This is a medical multiple-choice question written in {language}\n"
       "Question: {question}\nOptions: {options}\n"
       "Let's think step-by-step to reason about the correct answer based on medical knowledge. Write out your reasoning, "
       "then select the single best option (A, B, C, or D). End your reply with a final line of the form 'Answer: X', "
       "where X is the option letter.")


TEMPLATES = {'zs': ZS, 'cot': COT, 'umcot': UMCOT}


def fmt_options(opts):
    return '\n'.join(f'{k}: {v}' for k, v in sorted(opts.items()))


def build_prompt(item, script, cond):
    q = item[f'question_{script}']
    opts = item.get(f'options_{script}')
    if opts is None:
        raise KeyError(f'no options_{script}')
    return TEMPLATES[cond].format(language=LANG[script], question=q, options=fmt_options(opts))


# ---------------------------------------------------------------- answer parsing
PARSER_VERSION = 2


UR_LETTER = {'ا': 'A', 'اے': 'A', 'بی': 'B', 'ب': 'B', 'سی': 'C', 'ڈی': 'D', 'د': 'D'}


def parse_answer(text, valid, cond):
    """Return (letter or None, method)."""
    if not text:
        return None, 'empty'
    t = text.strip()
    valid = set(valid)
    pats = [r'(?:final\s+)?answer\s*(?:is)?\s*[:：\-]?\s*\**\(?([A-D])\)?\b',
            r'(?:jawab|javab|جواب)\s*[:：\-]?\s*\**\(?([A-D])\)?\b']
    if cond in ('cot', 'umcot'):
        lines = [l for l in t.splitlines() if l.strip()]
        last = lines[-1] if lines else ''
        for p in pats:
            m = re.search(p, last, re.I)
            if m and m.group(1).upper() in valid:
                return m.group(1).upper(), 'last_line'
        for p in pats:
            ms = list(re.finditer(p, t, re.I))
            if ms and ms[-1].group(1).upper() in valid:
                return ms[-1].group(1).upper(), 'answer_anywhere'
    core = re.sub(r'[\s\*\.\)\(:"`\'\[\]]', '', t)
    if len(core) > 1:  # model ignored 'only the letter': look at the final line, then explicit conclusions
        lines = [l for l in t.splitlines() if l.strip()]
        last_letters = set(re.findall(r'(?<![A-Za-z])([A-D])(?![A-Za-z])', lines[-1])) if lines else set()
        if len(last_letters) == 1 and next(iter(last_letters)) in valid:
            return last_letters.pop(), 'last_line_letter'
        concl = r"(?:correct\s+(?:option|answer|choice)\s+is|answer\s+is|let'?s\s+(?:output|select|choose|go\s+with)|i\s+(?:choose|select)|best\s+(?:option|answer)\s+is)\s*[:\-]?\s*\**\s*(?:option\s*)?\(?([A-D])\)?(?![A-Za-z])"
        ms = [m for m in re.finditer(concl, t, re.I) if m.group(1).upper() in valid]
        if ms:
            return ms[-1].group(1).upper(), 'conclusion_phrase'
    if core.upper() in valid and len(core) == 1:
        return core.upper(), 'bare_letter'
    m = re.match(r'^\W*(?:option\s*)?([A-D])\b', t, re.I)
    if m and m.group(1).upper() in valid and (len(t) < 40 or cond == 'zs'):
        return m.group(1).upper(), 'leading_letter'
    for p in pats:
        ms = list(re.finditer(p, t, re.I))
        if ms and ms[-1].group(1).upper() in valid:
            return ms[-1].group(1).upper(), 'answer_anywhere'
    if core in UR_LETTER and UR_LETTER[core] in valid:
        return UR_LETTER[core], 'urdu_letter'
    if cond == 'zs':
        letters = set(re.findall(r'\b([A-D])\b', t))
        if len(letters) == 1 and next(iter(letters)) in valid:
            return letters.pop(), 'single_letter_in_text'
    return None, 'unparsed'

#### 2. API clients with retries; hard stops on HTTP 402 and daily-quota 429

**Step 5: Define the API clients.** Functions that call the OpenAI and Gemini APIs, retry on temporary errors, and stop hard on billing (HTTP 402) or daily-quota errors so a run never fails silently or overspends.

*Input:* API keys (only used when `MODE = "run"`). *Output:* the functions `call_openai`, `call_gemini`, `call_with_retry`.

In [ ]:
# --- from scripts/run_eval.py (verbatim; the CLI scripts use the same code) ---
# ---------------------------------------------------------------- providers
class QuotaExhausted(Exception):
    pass


def call_gemini(args, prompt):
    url = f'https://generativelanguage.googleapis.com/v1beta/models/{args.model}:generateContent'
    gc = {'temperature': args.temperature, 'maxOutputTokens': args.max_tokens}
    if args.thinking_level:
        gc['thinkingConfig'] = {'thinkingLevel': args.thinking_level}
    elif args.thinking_budget is not None:
        gc['thinkingConfig'] = {'thinkingBudget': args.thinking_budget}
    body = {'contents': [{'role': 'user', 'parts': [{'text': prompt}]}], 'generationConfig': gc}
    r = requests.post(url, json=body, timeout=args.timeout,
                      headers={'x-goog-api-key': os.environ['GEMINI_API_KEY'], 'Content-Type': 'application/json'})
    d = r.json() if r.headers.get('content-type', '').startswith('application/json') else {'raw': r.text[:500]}
    if r.status_code != 200:
        return None, r.status_code, d
    cand = (d.get('candidates') or [{}])[0]
    parts = cand.get('content', {}).get('parts', [])
    text = ''.join(p.get('text', '') for p in parts if not p.get('thought'))
    return dict(text=text, finish_reason=cand.get('finishReason'), model_version=d.get('modelVersion'),
                response_id=d.get('responseId'), usage=d.get('usageMetadata')), 200, d


def call_openai(args, prompt):
    url = args.base_url.rstrip('/') + '/chat/completions'
    body = {'model': args.model, 'messages': [{'role': 'user', 'content': prompt}]}
    if not args.no_temperature:
        body['temperature'] = args.temperature
    body[args.max_tokens_param] = args.max_tokens
    if args.reasoning_effort:
        body['reasoning_effort'] = args.reasoning_effort
    key = os.environ.get(args.api_key_env, '')
    r = requests.post(url, json=body, timeout=args.timeout,
                      headers={'Authorization': f'Bearer {key}'} if key else {})
    try:
        d = r.json()
    except Exception:
        d = {'raw': r.text[:500]}
    if r.status_code != 200:
        return None, r.status_code, d
    ch = d['choices'][0]
    return dict(text=ch['message'].get('content') or '', finish_reason=ch.get('finish_reason'),
                model_version=d.get('model'), response_id=d.get('id'), usage=d.get('usage'),
                system_fingerprint=d.get('system_fingerprint'), service_tier=d.get('service_tier'),
                refusal=ch['message'].get('refusal')), 200, d


def retry_seconds(err):
    try:
        for det in err['error'].get('details', []):
            if det.get('@type', '').endswith('RetryInfo'):
                return float(det['retryDelay'].rstrip('s'))
    except Exception:
        pass
    return None


def is_daily(err):
    try:
        for det in err['error'].get('details', []):
            for v in det.get('violations', []):
                if 'PerDay' in v.get('quotaId', ''):
                    return True
    except Exception:
        pass
    return False


def call_with_retry(args, prompt):
    fn = call_gemini if args.provider == 'gemini' else call_openai
    delay = 5
    errors = []
    for attempt in range(args.max_retries):
        try:
            out, code, raw = fn(args, prompt)
        except requests.RequestException as e:
            out, code, raw = None, -1, {'exc': str(e)}
        if out is not None:
            out['errors'] = errors
            return out, attempt
        if code == 429 and is_daily(raw):
            with open(os.path.join(ROOT, 'quota_errors.log'), 'a') as qf:
                qf.write(dt.datetime.now(dt.timezone.utc).isoformat() + ' ' + json.dumps(raw) + '\n')
            raise QuotaExhausted(json.dumps(raw)[:600])
        if code == 402:
            raise QuotaExhausted('HTTP 402 (billing/prepayment): ' + json.dumps(raw)[:400])
        if code in (400, 401, 403, 404):
            raise SystemExit(f'non-retryable HTTP {code}: {json.dumps(raw)[:400]}')
        errors.append({'t': dt.datetime.now(dt.timezone.utc).isoformat(), 'code': code, 'body': json.dumps(raw)[:300]})
        wait = retry_seconds(raw) if code == 429 else None
        wait = max(wait or delay, 1) + random.uniform(0, 2)
        print(f'  [{code}] retry {attempt + 1}/{args.max_retries} in {wait:.0f}s: {json.dumps(raw)[:160]}', flush=True)
        time.sleep(wait)
        delay = min(delay * 2, 300)
    raise RuntimeError('max retries exceeded')

#### 3. Resumable runner
Appends one JSON line per call to `<outdir>/<model>/<script>_<cond>.jsonl` and skips items already done. It has an RPM throttle, worker threads, interleaved order and a cost cap. Called with the same arguments as the CLI.

**Step 6: Define the resumable API runner.** Loops over the items, builds each prompt, calls the model and appends one JSON line per call. Items already in the output file are skipped, so an interrupted run can be restarted. A cost cap stops it before it spends too much.

*Input:* the sample file and settings. *Output:* the function `run_eval_main`; when run it writes `<outdir>/<model>/<script>_<cond>.jsonl`.

In [ ]:
# --- from scripts/run_eval.py (verbatim; the CLI scripts use the same code) ---
# ---------------------------------------------------------------- main loop
def run_eval_main(argv=None):
    ap = argparse.ArgumentParser()
    ap.add_argument('--provider', choices=['gemini', 'openai'], default='gemini')
    ap.add_argument('--model', required=True)
    ap.add_argument('--base-url', default='https://api.openai.com/v1')
    ap.add_argument('--api-key-env', default='OPENAI_API_KEY')
    ap.add_argument('--items', default=os.path.join(ROOT, 'data/subset_items.jsonl'))
    ap.add_argument('--ur-options', default=os.path.join(ROOT, 'data/options_ur.jsonl'),
                    help='jsonl {uid, options_ur}; required for the Urdu script (not in the public U-MIRAGE release)')
    ap.add_argument('--ur-original-questions', action='store_true', help='use the released (truncated) Urdu questions')
    ap.add_argument('--scripts', nargs='+', default=['en', 'ru', 'ur'])
    ap.add_argument('--conds', nargs='+', default=['zs', 'cot'])
    ap.add_argument('--temperature', type=float, default=0.1)
    ap.add_argument('--max-tokens', type=int, default=4096)
    ap.add_argument('--thinking-level', default=None, help='Gemini 3+/4: minimal|low|medium|high')
    ap.add_argument('--thinking-budget', type=int, default=None, help='Gemini 2.5: 0 disables thinking on Flash')
    ap.add_argument('--no-temperature', action='store_true', help='OpenAI reasoning models that only allow default temperature')
    ap.add_argument('--max-tokens-param', default='max_completion_tokens', help='OpenAI: max_completion_tokens (new) or max_tokens (legacy servers)')
    ap.add_argument('--reasoning-effort', default=None, help='OpenAI-compatible reasoning models')
    ap.add_argument('--rpm', type=float, default=5, help='client-side throttle (requests per minute)')
    ap.add_argument('--max-calls', type=int, default=None, help='stop after this many successful calls (quota budgeting)')
    ap.add_argument('--limit', type=int, default=None, help='only the first N subset items (smoke test)')
    ap.add_argument('--interleave', action='store_true', help='round-robin across script/cond so partial runs stay balanced')
    ap.add_argument('--workers', type=int, default=1, help='concurrent requests (RPM throttle still applies)')
    ap.add_argument('--ids', default=None, help='csv with a uid column: restrict to these items (e.g. data/sample366_ids.csv)')
    ap.add_argument('--order', choices=['file', 'shuffle'], default='shuffle')
    ap.add_argument('--seed', type=int, default=42)
    ap.add_argument('--spend-this-model-only', action='store_true', help='count only this model\'s results toward the cost cap')
    ap.add_argument('--max-cost-usd', type=float, default=None, help='stop when estimated spend (all logs since --spend-since) exceeds this')
    ap.add_argument('--spend-since', default='2026-11-22T08:00', help='UTC ISO prefix; calls before this are treated as free tier')
    ap.add_argument('--price-in', type=float, default=0.30, help='USD per 1M input tokens')
    ap.add_argument('--price-out', type=float, default=2.50, help='USD per 1M output+thinking tokens')
    ap.add_argument('--timeout', type=int, default=180)
    ap.add_argument('--max-retries', type=int, default=8)
    ap.add_argument('--outdir', default=os.path.join(ROOT, 'results'))
    args = ap.parse_args(argv)

    items = [json.loads(l) for l in open(args.items)]
    if args.ids:
        import csv
        keep = {r['uid'] for r in csv.DictReader(open(args.ids))}
        items = [it for it in items if it['uid'] in keep]
    if args.limit:
        items = items[:args.limit]
    if args.order == 'shuffle':  # fixed-seed random order so partial runs are not biased by source
        import numpy as np
        perm = np.random.RandomState(args.seed).permutation(len(items))
        items = [items[i] for i in perm]
    if os.path.exists(args.ur_options):
        uo = {j['uid']: j['options_ur'] for j in map(json.loads, open(args.ur_options))}
        for it in items:
            if it['uid'] in uo:
                it['options_ur'] = uo[it['uid']]
        qf = os.path.join(ROOT, 'data/questions_ur_fixed.jsonl')
        if os.path.exists(qf) and not args.ur_original_questions:
            fx = {j['uid']: j['question_ur_fixed'] for j in map(json.loads, open(qf))}
            for it in items:
                if it['uid'] in fx:
                    it['question_ur'] = fx[it['uid']]; it['ur_question_fixed'] = True
            missing = [it['uid'] for it in items if it.get('ur_truncated', it.get('ur_maybe_truncated')) and not it.get('ur_question_fixed')]
            if missing and 'ur' in args.scripts:
                print(f'!! {len(missing)} truncated Urdu questions not yet fixed; skipping them for ur')
                for it in items:
                    if it['uid'] in missing: it.pop('options_ur', None)
    elif 'ur' in args.scripts:
        print(f'!! {args.ur_options} not found: Urdu-script options are not in the public U-MIRAGE release; skipping ur.')
        args.scripts = [s for s in args.scripts if s != 'ur']

    mdir = os.path.join(args.outdir, args.model.replace('/', '_'))
    os.makedirs(mdir, exist_ok=True)
    jobs = {}
    for s in args.scripts:
        for c in args.conds:
            path = os.path.join(mdir, f'{s}_{c}.jsonl')
            done = set()
            if os.path.exists(path):
                done = {json.loads(l)['uid'] for l in open(path) if l.strip()}
            jobs[(s, c)] = (path, [it for it in items if it['uid'] not in done and it.get(f'options_{s}')])
            print(f'{s}/{c}: {len(done)} done, {len(jobs[(s, c)][1])} to go')
    order = []
    if args.interleave:
        queues = {k: list(v[1]) for k, v in jobs.items()}
        while any(queues.values()):
            for k in list(queues):
                if queues[k]:
                    order.append((k, queues[k].pop(0)))
    else:
        order = [(k, it) for k, v in jobs.items() for it in v[1]]

    def cost_of(u):
        u = u or {}
        if 'prompt_tokens' in u:  # OpenAI: completion_tokens already include reasoning tokens
            return (u.get('prompt_tokens', 0) * args.price_in + u.get('completion_tokens', 0) * args.price_out) / 1e6
        return (u.get('promptTokenCount', 0) * args.price_in + (u.get('candidatesTokenCount', 0) + u.get('thoughtsTokenCount', 0)) * args.price_out) / 1e6
    spent = 0.0
    if args.max_cost_usd is not None:
        if args.spend_this_model_only:
            logs = glob.glob(os.path.join(mdir, '*.jsonl'))
        else:
            logs = glob.glob(os.path.join(args.outdir, '*', '*.jsonl')) + [os.path.join(ROOT, 'data/urdu_fix_log.jsonl')]
        for lf in logs:
            if os.path.exists(lf):
                for l in open(lf):
                    try: r = json.loads(l)
                    except Exception: continue
                    if r.get('t_start', '') >= args.spend_since:
                        spent += cost_of(r.get('usage'))
        print(f'estimated spend so far since {args.spend_since}Z: ${spent:.3f} (cap ${args.max_cost_usd})', flush=True)
    import threading
    from concurrent.futures import ThreadPoolExecutor
    gap = 60.0 / args.rpm
    lock, wlock, stop = threading.Lock(), threading.Lock(), threading.Event()
    state = {'next': 0.0, 'ok': 0, 'issued': 0, 'spent': spent}

    def work(job):
        (s, c), it = job
        if stop.is_set():
            return
        if args.max_cost_usd is not None and state['spent'] >= args.max_cost_usd:
            if not stop.is_set(): print(f'!! cost cap reached (${state["spent"]:.3f}), stopping', flush=True)
            stop.set(); return
        with lock:  # shared client-side RPM throttle + call budget
            if args.max_calls is not None and state['issued'] >= args.max_calls:
                stop.set(); return
            state['issued'] += 1
            slot = max(time.time(), state['next']); state['next'] = slot + gap
        if slot > time.time():
            time.sleep(slot - time.time())
        if stop.is_set():
            return
        prompt = build_prompt(it, s, c)
        t0 = dt.datetime.now(dt.timezone.utc)
        try:
            out, retries = call_with_retry(args, prompt)
        except QuotaExhausted as e:
            if not stop.is_set():
                print('!! daily quota exhausted, stopping (rerun later to resume):', str(e)[:300], flush=True)
            stop.set(); return
        except (RuntimeError, SystemExit) as e:
            print(f'!! {s}/{c} {it["uid"]} failed: {e}', flush=True); return
        pred, how = parse_answer(out['text'], it['options_en'].keys(), c)
        rec = dict(uid=it['uid'], source=it['source'], script=s, cond=c, gold=it['answer'], pred=pred,
                   correct=pred == it['answer'], parse=how, ur_question_fixed=it.get('ur_question_fixed', False) if s == 'ur' else None, response=out['text'], finish_reason=out['finish_reason'],
                   model=args.model, model_version=out['model_version'], response_id=out['response_id'],
                   usage=out['usage'], temperature=args.temperature, thinking_level=args.thinking_level,
                   thinking_budget=args.thinking_budget, retries=retries, prompt=prompt,
                   provider=args.provider, reasoning_effort=args.reasoning_effort,
                   temperature_sent=None if (args.provider == 'openai' and args.no_temperature) else args.temperature,
                   max_tokens=args.max_tokens, errors=out.get('errors'), system_fingerprint=out.get('system_fingerprint'),
                   service_tier=out.get('service_tier'), refusal=out.get('refusal'),
                   ur_question_truncated_in_original=it.get('ur_question_truncated_in_original', it.get('ur_truncated')) if s == 'ur' else None,
                   latency_s=round((dt.datetime.now(dt.timezone.utc) - t0).total_seconds(), 3),
                   t_start=t0.isoformat(), t_end=dt.datetime.now(dt.timezone.utc).isoformat())
        with wlock:
            with open(jobs[(s, c)][0], 'a') as f:
                f.write(json.dumps(rec, ensure_ascii=False) + '\n')
            state['ok'] += 1
            state['spent'] += cost_of(out['usage'])
            u = out['usage'] or {}
            print(f'{state["ok"]} {s}/{c} {it["uid"]} gold={it["answer"]} pred={pred} ({how}) '
                  f'tok={u.get("totalTokenCount", u.get("total_tokens"))}', flush=True)

    with ThreadPoolExecutor(max_workers=args.workers) as ex:
        list(ex.map(work, order))
    n_ok = state['ok']
    print('done; successful calls this session:', n_ok, f'| est. spend since {args.spend_since}Z: ${state["spent"]:.3f}')

#### 4. Scorers
`score_main`: accuracy, source-weighted accuracy, paired gaps (bootstrap 95% CI, exact McNemar), per-dataset accuracy, reply script.

**Step 7: Define the main scorer.** Re-parses every raw reply and computes accuracy, paired EN-RU / EN-UR / RU-UR gaps with 95% bootstrap intervals and exact McNemar tests, accuracy per source dataset, and which script the model replied in.

*Input:* raw `.jsonl` results. *Output:* the function `score_main`; when run it writes `<model>/scores_sample366.md`.

In [ ]:
# --- from scripts/score.py (verbatim; the CLI scripts use the same code) ---
import argparse, glob, json, os, re


import numpy as np, sys


from scipy.stats import binomtest


RU_WORDS = set('hai hain ka ki ke mein se aur nahi nahin kya yeh ye woh wo ko ho hota hoti hote jata jati liye bhi tha thi sakta sakti kar karta karti par lekin kyunke kyun agar jo ek sab zyada kam wala wali'.split())


EN_WORDS = set('the is are of and to in that with for this which it be as by on not or an was'.split())


def reply_script(text):
    t = re.sub(r'(?im)^\s*answer\s*:.*$', '', text or '')
    ar = len(re.findall(r'[\u0600-\u06FF]', t)); la = len(re.findall(r'[A-Za-z]', t))
    if ar + la == 0:
        return 'none'
    if ar / (ar + la) > 0.5:
        return 'ur'
    w = re.findall(r'[a-z]+', t.lower())
    ru, en = sum(x in RU_WORDS for x in w), sum(x in EN_WORDS for x in w)
    return 'ru' if ru > en else 'en'


def boot_ci(diff, B=10000, seed=0):
    rng = np.random.default_rng(seed); n = len(diff)
    m = diff[rng.integers(0, n, size=(B, n))].mean(1)
    return np.percentile(m, [2.5, 97.5])


def mcnemar(a, b):
    b01, b10 = int(((a == 1) & (b == 0)).sum()), int(((a == 0) & (b == 1)).sum())
    p = binomtest(b01, b01 + b10, 0.5).pvalue if b01 + b10 else 1.0
    return b01, b10, p


def score_main(argv=None):
    ap = argparse.ArgumentParser(); ap.add_argument('--model', required=True)
    ap.add_argument('--ids', default=None, help='csv with a uid column: score only these items')
    ap.add_argument('--outdir', default=os.path.join(ROOT, 'results'))
    args = ap.parse_args(argv)
    mdir = os.path.join(args.outdir, args.model.replace('/', '_'))
    SUBSRC = {j['uid']: j['source'] for j in map(json.loads, open(os.path.join(ROOT, 'data/subset_items.jsonl')))}
    VALID = {j['uid']: sorted(j['options_en']) for j in map(json.loads, open(os.path.join(ROOT, 'data/subset_items.jsonl')))}
    KEEP = None
    if args.ids:
        import csv
        KEEP = {r['uid'] for r in csv.DictReader(open(args.ids))}
        SUBSRC = {u: v for u, v in SUBSRC.items() if u in KEEP}
    R = {}
    for f in sorted(glob.glob(os.path.join(mdir, '*_*.jsonl'))):
        s, c = os.path.basename(f)[:-6].split('_', 1)
        recs = {}
        for l in open(f):
            r = json.loads(l)
            valid = VALID.get(r['uid']) or 'ABCD'  # re-parse raw reply with current parser
            r['pred'], r['parse'] = parse_answer(r['response'], valid, r['cond']); r['correct'] = r['pred'] == r['gold']
            if KEEP is None or r['uid'] in KEEP:
                recs[r['uid']] = r  # last write wins
        R[(s, c)] = recs
    out = [f'# Scores for {args.model} (answers re-parsed from raw replies, parser v{PARSER_VERSION})\n']
    out.append('| script | cond | n | accuracy | source-weighted acc* | parse fail | zs non-compliant (not a bare letter) | mean in tok | mean out tok (visible+thinking) |\n|---|---|---|---|---|---|---|---|---|')
    for (s, c), recs in sorted(R.items()):
        v = list(recs.values()); n = len(v)
        acc = np.mean([r['correct'] for r in v]); pf = sum(r['pred'] is None for r in v)
        ui = np.mean([(r['usage'] or {}).get('promptTokenCount', (r['usage'] or {}).get('prompt_tokens', 0)) for r in v])
        uo = np.mean([(r['usage'] or {}).get('candidatesTokenCount', 0) + (r['usage'] or {}).get('thoughtsTokenCount', 0)
                      + (r['usage'] or {}).get('completion_tokens', 0) for r in v])  # OpenAI usage keys too
        nc = sum(len(re.sub(r'[\s\*\.\(\)]', '', r['response'] or '')) > 1 for r in v) if c == 'zs' else None
        nc_s = f'{nc} ({100*nc/n:.1f}%)' if nc is not None else '-'
        W = {so: sum(1 for x in SUBSRC.values() if x == so) / len(SUBSRC) for so in set(SUBSRC.values())}
        per = {so: np.mean([r['correct'] for r in v if r['source'] == so]) for so in W if sum(r['source'] == so for r in v) >= 10}
        wacc = sum(W[so] * a for so, a in per.items()) / sum(W[so] for so in per) if per else float('nan')
        out.append(f'| {s} | {c} | {n} | {100*acc:.1f} | {100*wacc:.1f} | {pf} ({100*pf/n:.1f}%) | {nc_s} | {ui:.0f} | {uo:.0f} |')
    out.append('\n*re-weighted to the subset source mix (over sources with >=10 items done), to offset processing-order imbalance.')
    out.append('\n## Paired comparisons (items answered in both; diff = first - second, pp)\n')
    out.append('| cond | pair | n paired | acc 1st | acc 2nd | diff | 95% bootstrap CI (10k) | b01/b10 | McNemar exact p |\n|---|---|---|---|---|---|---|---|---|')
    for c in sorted({c for _, c in R}):
        for a, b in [('en', 'ur'), ('en', 'ru'), ('ru', 'ur')]:
            if (a, c) not in R or (b, c) not in R:
                continue
            ids = sorted(set(R[(a, c)]) & set(R[(b, c)]))
            if len(ids) < 2:
                continue
            x = np.array([R[(a, c)][i]['correct'] for i in ids], int); y = np.array([R[(b, c)][i]['correct'] for i in ids], int)
            lo, hi = boot_ci(x - y); b01, b10, p = mcnemar(x, y)
            out.append(f'| {c} | {a.upper()}-{b.upper()} | {len(ids)} | {100*x.mean():.1f} | {100*y.mean():.1f} | {100*(x-y).mean():+.1f} | [{100*lo:+.1f}, {100*hi:+.1f}] | {b01}/{b10} | {p:.3g} |')
    out.append('\n## Accuracy by source dataset (%; n in brackets)\n')
    srcs = sorted({r['source'] for recs in R.values() for r in recs.values()})
    out.append('| script/cond | ' + ' | '.join(srcs) + ' |\n|---|' + '---|' * len(srcs))
    for (s, c), recs in sorted(R.items()):
        cells = []
        for so in srcs:
            v = [r['correct'] for r in recs.values() if r['source'] == so]
            cells.append(f'{100*np.mean(v):.1f} [{len(v)}]' if v else '-')
        out.append(f'| {s}/{c} | ' + ' | '.join(cells) + ' |')
    cot = [(k, v) for k, v in R.items() if k[1] in ('cot', 'umcot')]
    if cot:
        out.append('\n## Reply script for reasoning outputs (heuristic: Arabic-script share; Roman Urdu vs English by function words)\n')
        out.append('| asked | n | replied en | replied ru | replied ur | % in asked script |\n|---|---|---|---|---|---|')
        for (s, c), recs in sorted(cot):
            sc = [reply_script(r['response']) for r in recs.values()]; n = len(sc)
            out.append(f'| {s}/{c} | {n} | {sc.count("en")} | {sc.count("ru")} | {sc.count("ur")} | {100*sc.count(s)/n:.0f}% |')
    txt = '\n'.join(out) + '\n'
    open(os.path.join(mdir, 'scores' + ('_' + os.path.basename(args.ids).split('_')[0] if args.ids else '') + '.md'), 'w').write(txt); print(txt)

`score_extra_main`: paper-style table, % table, gaps, truncated-vs-complete Urdu split, compliance, U-MIRAGE paper reference rows.

**Step 8: Define the paper-style tables.** Builds the U-MIRAGE Table 4-style accuracy table (with the paper's reference rows), the gap table, the truncated vs complete Urdu question split, and answer-format compliance.

*Input:* raw `.jsonl` results. *Output:* the function `score_extra_main`; writes `<model>/scores_extra_sample366.md` and a combined `summary_*.md`.

In [ ]:
# --- from scripts/score_extra.py (verbatim; the CLI scripts use the same code) ---
import argparse, json, os, re, sys, csv


import numpy as np


CONDS = [('zs', 'Zero-Shot'), ('umcot', 'CoT (U-MIRAGE prompt)')]


SCRIPTS = ['en', 'ru', 'ur']


# U-MIRAGE paper reference (umirage/all_configurations_accuracy.csv; full benchmark, "CoT" = their with_reasoning prompt)
PAPER_ALL = {  # U-MIRAGE all_configurations_accuracy.csv, full 7,663-item benchmark ("CoT" = their with_reasoning prompt)
    'GPT-5 nano [paper]':            {('en', 'zs'): .7962, ('en', 'umcot'): .7865, ('ru', 'zs'): .7502, ('ru', 'umcot'): .7498, ('ur', 'zs'): .7299, ('ur', 'umcot'): .7023},
    'Gemini 2.0 Flash-Lite [paper]': {('en', 'zs'): .7177, ('en', 'umcot'): .7779, ('ru', 'zs'): .6582, ('ru', 'umcot'): .7414, ('ur', 'zs'): .6768, ('ur', 'umcot'): .7013},
    'Gemma 2 2B [paper]':            {('en', 'zs'): .4198, ('en', 'umcot'): .4325, ('ru', 'zs'): .3247, ('ru', 'umcot'): .3500, ('ur', 'zs'): .3124, ('ur', 'umcot'): .3367},
    'Qwen 2.5 0.5B (q) [paper]':     {('en', 'zs'): .2592, ('en', 'umcot'): .2974, ('ru', 'zs'): .2378, ('ru', 'umcot'): .2765, ('ur', 'zs'): .2305, ('ur', 'umcot'): .2721},
    'LLaMA 3.2 1B (q) [paper]':      {('en', 'zs'): .1886, ('en', 'umcot'): .3889, ('ru', 'zs'): .1982, ('ru', 'umcot'): .2400, ('ur', 'zs'): .1883, ('ur', 'umcot'): .1781},
    'LLaMA 3.2 1B [paper]':          {('en', 'zs'): .4549, ('en', 'umcot'): .4437, ('ru', 'zs'): .3448, ('ru', 'umcot'): .3616, ('ur', 'zs'): .3364, ('ur', 'umcot'): .3124},
}


PAPER_SETS = {'small': ['Qwen 2.5 0.5B (q) [paper]', 'LLaMA 3.2 1B (q) [paper]', 'LLaMA 3.2 1B [paper]'],
              'api': ['GPT-5 nano [paper]', 'Gemini 2.0 Flash-Lite [paper]', 'Gemma 2 2B [paper]'],
              'all': list(PAPER_ALL)}


# which U-MIRAGE paper row each of our runs succeeds (for the side-by-side table)
SUCCESSOR_OF = {'gpt-6-luna': 'GPT-5 nano [paper]', 'gemini-3.5-flash-lite': 'Gemini 2.0 Flash-Lite [paper]',
                'gemma-4-26b-a4b-it': 'Gemma 2 2B [paper]', 'qwen3.5-0.8b-bnb4': 'Qwen 2.5 0.5B (q) [paper]',
                'llama3.2-1b-instruct-bnb4': 'LLaMA 3.2 1B (q) [paper]', 'llama3.2-1b-instruct-bf16': 'LLaMA 3.2 1B [paper]'}


def load(mdir, keep):
    R = {}
    for s in SCRIPTS:
        for c, _ in CONDS:
            f = os.path.join(mdir, f'{s}_{c}.jsonl')
            if not os.path.exists(f):
                continue
            recs = {}
            for l in open(f):
                if not l.strip():
                    continue
                r = json.loads(l)
                if keep and r['uid'] not in keep:
                    continue
                r['pred'], r['parse'] = parse_answer(r['response'], 'ABCD' if r['gold'] in 'ABCD' else r['gold'], r['cond'])
                r['correct'] = r['pred'] == r['gold']
                recs[r['uid']] = r
            R[(s, c)] = recs
    return R


def paired(R, a, b, c, ids=None):
    if (a, c) not in R or (b, c) not in R:
        return None
    common = set(R[(a, c)]) & set(R[(b, c)])
    if ids is not None:
        common &= ids
    common = sorted(common)
    if len(common) < 2:
        return None
    x = np.array([R[(a, c)][i]['correct'] for i in common], int); y = np.array([R[(b, c)][i]['correct'] for i in common], int)
    lo, hi = boot_ci(x - y); b01, b10, p = mcnemar(x, y)
    return len(common), x.mean(), y.mean(), (x - y).mean(), lo, hi, b01, b10, p


def score_extra_main(argv=None):
    ap = argparse.ArgumentParser()
    ap.add_argument('--models', nargs='+', required=True)
    ap.add_argument('--labels', nargs='*', default=None, help='display names, same order as --models')
    ap.add_argument('--outdir', default=os.path.join(ROOT, 'results'))
    ap.add_argument('--items', default=os.path.join(ROOT, 'share/nlp_project_sample366.jsonl'))
    ap.add_argument('--ids', default=os.path.join(ROOT, 'data/sample366_ids.csv'))
    ap.add_argument('--paper-rows', choices=list(PAPER_SETS), default='small')
    ap.add_argument('--summary-name', default='summary_small_models_sample366.md')
    ap.add_argument('--title', default='Small-model successors on the 366-item sample')
    args = ap.parse_args(argv)
    items = {j['uid']: j for j in map(json.loads, open(args.items))}
    keep = {r['uid'] for r in csv.DictReader(open(args.ids))} if args.ids and os.path.exists(args.ids) else set(items)
    trunc = {u for u, j in items.items() if j.get('ur_question_truncated_in_original', j.get('ur_truncated'))}
    labels = args.labels or args.models
    summary = [f'# {args.title} (parser v{PARSER_VERSION})\n',
               '## Paper-style accuracy table (fraction correct, 2 decimals; n per cell in the per-model files)\n',
               '| Model | EN Zero-Shot | EN CoT | RU Zero-Shot | RU CoT | UR Zero-Shot | UR CoT |', '|---|---|---|---|---|---|---|']
    gaps_rows, trunc_rows, comp_rows, pct_rows, pct_of = [], [], [], [], {}
    for tag, lab in zip(args.models, labels):
        mdir = os.path.join(args.outdir, tag)
        R = load(mdir, keep)
        out = [f'# Extra scores for {lab} ({tag}), parser v{PARSER_VERSION}\n',
               '| script | cond | n | accuracy | 95% bootstrap CI | parse fail | zs not bare letter | hit max_tokens |', '|---|---|---|---|---|---|---|---|']
        cells = []
        for s in SCRIPTS:
            for c, _ in CONDS:
                v = list(R.get((s, c), {}).values())
                if not v:
                    cells.append('-'); continue
                x = np.array([r['correct'] for r in v], float)
                rng = np.random.default_rng(0); bs = x[rng.integers(0, len(x), size=(10000, len(x)))].mean(1)
                lo, hi = np.percentile(bs, [2.5, 97.5])
                pf = sum(r['pred'] is None for r in v)
                nb = sum(len(re.sub(r'[\s\*\.\(\)]', '', r['response'] or '')) > 1 for r in v) if c == 'zs' else None
                ln = sum(r.get('finish_reason') in ('length', 'MAX_TOKENS') for r in v)
                out.append(f'| {s} | {c} | {len(v)} | {x.mean():.2f} | [{lo:.2f}, {hi:.2f}] | {pf} ({100*pf/len(v):.1f}%) | '
                           f'{"-" if nb is None else f"{nb} ({100*nb/len(v):.1f}%)"} | {ln} ({100*ln/len(v):.1f}%) |')
                cells.append(f'{x.mean():.2f}' + ('' if len(v) == len(keep) else f' (n={len(v)})'))
                comp_rows.append(f'| {lab} | {s}/{c} | {len(v)} | {pf} ({100*pf/len(v):.1f}%) | {"-" if nb is None else f"{100*nb/len(v):.1f}%"} | {100*ln/len(v):.1f}% |')
        summary.append(f'| {lab} | ' + ' | '.join(cells) + ' |')
        pct_of[tag] = [f'{100*np.mean([r["correct"] for r in R[(s, c)].values()]):.2f}' if R.get((s, c)) else '-' for s in SCRIPTS for c, _ in CONDS]
        pct_rows.append(f'| {lab} | ' + ' | '.join(pct_of[tag]) + ' |')
        out += ['\n## Paired gaps (accuracy in %, diff = first - second in percentage points; bootstrap 10k; exact McNemar)\n',
                '| cond | pair | n | acc 1st | acc 2nd | diff | 95% CI | b01/b10 | p |', '|---|---|---|---|---|---|---|---|---|']
        for c, cn in CONDS:
            for a, b in [('en', 'ru'), ('en', 'ur'), ('ru', 'ur')]:
                g = paired(R, a, b, c)
                if g:
                    n, xa, xb, d, lo, hi, b01, b10, p = g
                    row = f'| {c} | {a.upper()}-{b.upper()} | {n} | {100*xa:.2f} | {100*xb:.2f} | {100*d:+.2f} | [{100*lo:+.2f}, {100*hi:+.2f}] | {b01}/{b10} | {p:.3g} |'
                    out.append(row); gaps_rows.append(f'| {lab} ' + row)
        out += ['\n## Urdu: questions truncated in the released U-MIRAGE CSV vs complete (original questions used)\n',
                '| cond | subset | n | UR acc | EN acc (same items) | EN-UR | 95% CI | McNemar p |', '|---|---|---|---|---|---|---|---|']
        for c, cn in CONDS:
            for nm, ids in [('truncated', trunc & keep), ('complete', keep - trunc)]:
                g = paired(R, 'en', 'ur', c, ids)
                if g:
                    n, xa, xb, d, lo, hi, b01, b10, p = g
                    row = f'| {c} | {nm} | {n} | {100*xb:.2f} | {100*xa:.2f} | {100*d:+.2f} | [{100*lo:+.2f}, {100*hi:+.2f}] | {p:.3g} |'
                    out.append(row); trunc_rows.append(f'| {lab} ' + row)
        txt = '\n'.join(out) + '\n'
        open(os.path.join(mdir, 'scores_extra_sample366.md'), 'w').write(txt)
        print(txt)
    summary.append('| *U-MIRAGE paper reference rows (full benchmark, 7,663 items):* | | | | | | |')
    for lab in PAPER_SETS[args.paper_rows]:
        d = PAPER_ALL[lab]
        summary.append(f'| {lab} | ' + ' | '.join(f'{d[(s, c)]:.2f}' for s in SCRIPTS for c, _ in CONDS) + ' |')
    summary += ['\n## Accuracy in % (2 decimals)\n', '| Model | EN ZS | EN CoT | RU ZS | RU CoT | UR ZS | UR CoT |', '|---|---|---|---|---|---|---|'] + pct_rows
    pairs = [(SUCCESSOR_OF[t], t, lab) for t, lab in zip(args.models, labels) if SUCCESSOR_OF.get(t) in PAPER_SETS[args.paper_rows]]
    order = list(PAPER_ALL)
    pairs.sort(key=lambda x: order.index(x[0]))
    if pairs:
        summary += ['\n## Paper rows vs our successors (accuracy in %, 2 decimals; each paper row directly above its successor)\n',
                    'Paper = U-MIRAGE, full benchmark (7,663 items). Ours = a random 366-item sample (seed 42), so the rows are not item-matched. '
                    'Gemma 4 26B-A4B is not size-matched to Gemma 2 2B (26B total / ~4B active parameters vs 2B).\n',
                    '| Model | EN ZS | EN CoT | RU ZS | RU CoT | UR ZS | UR CoT |', '|---|---|---|---|---|---|---|']
        for prow, t, lab in pairs:
            summary.append(f'| {prow.replace(" [paper]", "")} [paper] | ' + ' | '.join(f'{100*PAPER_ALL[prow][(s, c)]:.2f}' for s in SCRIPTS for c, _ in CONDS) + ' |')
            summary.append(f'| {lab} [ours] | ' + ' | '.join(pct_of[t]) + ' |')
    summary += ['\n## Paired gaps (accuracy %, diff in pp)\n', '| model | cond | pair | n | acc 1st | acc 2nd | diff | 95% CI | b01/b10 | McNemar p |',
                '|---|---|---|---|---|---|---|---|---|---|'] + gaps_rows
    summary += ['\n## Truncated vs complete Urdu questions (accuracy %, diff in pp)\n', '| model | cond | subset | n | UR acc | EN acc | EN-UR | 95% CI | McNemar p |',
                '|---|---|---|---|---|---|---|---|---|'] + trunc_rows
    summary += ['\n## Compliance\n', '| model | run | n | parse fail | zs not bare letter | hit max_tokens |', '|---|---|---|---|---|---|'] + comp_rows
    txt = '\n'.join(summary) + '\n'
    open(os.path.join(args.outdir, args.summary_name), 'w').write(txt)
    print(txt)

`reproduce_main`: regenerates all tables and independently checks the key numbers against `expected_results.json`.

**Step 9: Define the reproduction check.** Regenerates every results table for all six models and independently recomputes the key numbers, then compares them with `expected_results.json`. It prints PASS only if everything matches.

*Input:* the included `results/` and `expected_results.json`. *Output:* the function `reproduce_main`.

In [ ]:
# --- from scripts/reproduce_tables.py (verbatim; the CLI scripts use the same code) ---
import argparse, contextlib, io, json, os, sys


import numpy as np


REPRO_API = {'gpt-6-luna': 'GPT-6 Luna', 'gemma-4-26b-a4b-it': 'Gemma 4 26B-A4B', 'gemini-3.5-flash-lite': 'Gemini 3.5 Flash-Lite'}


REPRO_SMALL = {'qwen3.5-0.8b-bnb4': 'Qwen3.5 0.8B (q, NF4)', 'llama3.2-1b-instruct-bnb4': 'Llama 3.2 1B Instruct (q, NF4)',
               'llama3.2-1b-instruct-bf16': 'Llama 3.2 1B Instruct (bf16)'}


REPRO_MODELS = {**REPRO_API, **REPRO_SMALL}  # all six models


REPRO_SCRIPTS, REPRO_CONDS = ['en', 'ru', 'ur'], ['zs', 'umcot']


def metrics(rdir):
    items = {j['uid']: j for j in map(json.loads, open(os.path.join(ROOT, 'share/nlp_project_sample366.jsonl')))}
    trunc = {u for u, j in items.items() if j.get('ur_question_truncated_in_original')}
    out = {'parser_version': PARSER_VERSION, 'models': {}}
    for m in REPRO_MODELS:
        R = {}
        for s in REPRO_SCRIPTS:
            for c in REPRO_CONDS:
                recs = {}
                for l in open(os.path.join(rdir, m, f'{s}_{c}.jsonl')):
                    r = json.loads(l)
                    if r['uid'] in items:
                        pred, _ = parse_answer(r['response'], 'ABCD', r['cond'])
                        recs[r['uid']] = (pred == r['gold'], pred is None)
                R[(s, c)] = recs
        mm = {'accuracy_pct': {}, 'n': {}, 'parse_fail': {}, 'gaps': {}, 'ur_truncation_split': {}}
        for (s, c), recs in R.items():
            mm['n'][f'{s}_{c}'] = len(recs)
            mm['accuracy_pct'][f'{s}_{c}'] = round(100 * np.mean([v[0] for v in recs.values()]), 2)
            mm['parse_fail'][f'{s}_{c}'] = int(sum(v[1] for v in recs.values()))
        for c in REPRO_CONDS:
            for a, b in [('en', 'ru'), ('en', 'ur'), ('ru', 'ur')]:
                ids = sorted(set(R[(a, c)]) & set(R[(b, c)]))
                x = np.array([R[(a, c)][i][0] for i in ids], int); y = np.array([R[(b, c)][i][0] for i in ids], int)
                b01, b10, p = mcnemar(x, y)
                mm['gaps'][f'{c}_{a}-{b}'] = {'n': len(ids), 'diff_pp': round(100 * (x - y).mean(), 2), 'b01': b01, 'b10': b10, 'mcnemar_p': float(f'{p:.4g}')}
            for nm, sub in [('truncated', trunc), ('complete', set(items) - trunc)]:
                ids = sorted(set(R[('ur', c)]) & sub)
                mm['ur_truncation_split'][f'{c}_{nm}'] = {'n': len(ids), 'ur_acc_pct': round(100 * np.mean([R[('ur', c)][i][0] for i in ids]), 2),
                                                          'en_acc_pct': round(100 * np.mean([R[('en', c)][i][0] for i in ids]), 2)}
        out['models'][m] = mm
    return out


def compare(a, b, path=''):
    bad = []
    if isinstance(a, dict):
        for k in set(a) | set(b):
            if k not in a or k not in b:
                bad.append(f'{path}/{k}: missing'); continue
            bad += compare(a[k], b[k], f'{path}/{k}')
    elif isinstance(a, float) or isinstance(b, float):
        if not np.isclose(a, b, rtol=1e-3, atol=1e-9):
            bad.append(f'{path}: got {a}, expected {b}')
    elif a != b:
        bad.append(f'{path}: got {a}, expected {b}')
    return bad


def reproduce_main(argv=None):
    ap = argparse.ArgumentParser()
    ap.add_argument('--results', default=os.path.join(ROOT, 'results'))
    ap.add_argument('--write-expected', action='store_true')
    args = ap.parse_args(argv)
    # 1) regenerate the markdown tables with the project's scorers
    with contextlib.redirect_stdout(io.StringIO()):  # the scorers print their tables; keep this check's output short
        for m in REPRO_MODELS:
            score_main(['--model', m, '--outdir', args.results, '--ids', os.path.join(ROOT, 'data/sample366_ids.csv')])
        for group, rows, name, title in [(REPRO_API, 'api', 'summary_api_models_sample366.md', 'API models on the 366-item sample'),
                                         (REPRO_SMALL, 'small', 'summary_small_models_sample366.md', 'Small-model successors on the 366-item sample'),
                                         (REPRO_MODELS, 'all', 'summary_all_models_sample366.md', 'All six models on the 366-item sample')]:
            score_extra_main(['--outdir', args.results, '--models', *group, '--labels', *group.values(), '--paper-rows', rows,
                              '--summary-name', name, '--title', title])
    print('regenerated: results/<model>/scores_sample366.md, results/<model>/scores_extra_sample366.md (6 models), '
          'results/summary_{api,small,all}_models_sample366.md')
    # 2) recompute key numbers independently and compare with the reported values
    got = metrics(args.results)
    exp_path = os.path.join(ROOT, 'expected_results.json')
    if args.write_expected:
        json.dump(got, open(exp_path, 'w'), indent=1); print('wrote', exp_path); return
    bad = compare(got, json.load(open(exp_path)))
    print('\nAccuracy (%)                   ' + '  '.join(f'{s.upper()}-{c:<5}' for s in REPRO_SCRIPTS for c in REPRO_CONDS))
    for m, lab in REPRO_MODELS.items():
        print(f'{lab:<31}' + '  '.join(f'{got["models"][m]["accuracy_pct"][f"{s}_{c}"]:>8.2f}' for s in REPRO_SCRIPTS for c in REPRO_CONDS))
    if bad:
        print(f'\nFAIL: {len(bad)} mismatches'); [print('  ', b) for b in bad[:50]]; sys.exit(1)
    print(f'\nPASS: all reproduced numbers match expected_results.json (accuracies, n, parse failures, gaps, McNemar counts/p, truncation split)')

## (a) Score only (default): reproduce the reported tables, no API calls

**Step 10: Score only: check our numbers (default).** Runs the reproduction check on the included raw outputs. No keys and no network are needed. The last line should say PASS.

*Input:* the included `results/`. *Output:* regenerated score files in `results/` and a PASS / FAIL line.

In [ ]:
if MODE == 'score_only':
    reproduce_main([])      # raises SystemExit(1) and lists mismatches if anything differs

## (b) Re-run the models (`MODE = "run"`)

**Step 11: Load API keys (run mode only).** Reads `OPENAI_API_KEY` / `GEMINI_API_KEY` from Colab secrets or environment variables. Keys are never printed or saved. Skipped in score-only mode.

*Input:* Colab secrets or environment variables. *Output:* keys in the process environment only.

In [ ]:
# ---- API keys; never printed ----
def load_key(name):
    if os.environ.get(name):
        return True
    if IN_COLAB:
        try:
            from google.colab import userdata
            value = userdata.get(name)
            if value:
                os.environ[name] = value
                return True
        except Exception:
            pass
    return False

if MODE == 'run':
    for k, needed in {'OPENAI_API_KEY': any(m.startswith('gpt') for m in RUN_MODELS),
                      'GEMINI_API_KEY': any(not m.startswith('gpt') for m in RUN_MODELS)}.items():
        if needed:
            print(k, 'found' if load_key(k) else 'MISSING (add it as a Colab secret or environment variable)')

**Step 12: Re-run the API models (run mode only).** Calls each model with exactly the settings we used (reasoning level, temperature, token limit, rate limit, cost cap). Re-running this cell continues where it stopped. In score-only mode it does nothing.

*Input:* keys, the sample file. *Output:* `results_rerun/<model>/*.jsonl` (or `smoke_results/` for a smoke test) and a log per model.

In [ ]:
# ---- exact settings per model (one runner call per entry); resumable: re-run this cell to continue ----
import contextlib, datetime as dt, io
COMMON = ['--items', os.path.join(DATA_DIR, 'share/nlp_project_sample366.jsonl'), '--ur-original-questions',
          '--scripts', 'en', 'ru', 'ur', '--interleave', '--order', 'shuffle', '--seed', '42',
          '--temperature', '0.1', '--max-tokens', '2048']
CMDS = {
    'gpt-6-luna': [('luna', ['--provider', 'openai', '--base-url', 'https://api.openai.com/v1', '--model', 'gpt-6-luna',
                             '--reasoning-effort', 'none', '--rpm', '900', '--workers', '16',
                             '--price-in', '0.10', '--price-out', '0.50', '--max-cost-usd', '3', '--conds', 'zs', 'umcot'])],
    'gemma-4-26b-a4b-it': [(f'gemma_{c}', ['--provider', 'gemini', '--model', 'gemma-4-26b-a4b-it', '--thinking-level', 'minimal',
                                           '--rpm', '28', '--workers', '6', '--timeout', '120', '--price-in', '0.15',
                                           '--price-out', '0.60', '--max-cost-usd', '2', '--conds', c]) for c in ('zs', 'umcot')],
    'gemini-3.5-flash-lite': [(f'fl35_{c}', ['--provider', 'gemini', '--model', 'gemini-3.5-flash-lite', '--thinking-level', 'minimal',
                                             '--rpm', '60', '--workers', '4', '--price-in', '0.30', '--price-out', '2.50',
                                             '--max-cost-usd', '2', '--conds', c]) for c in ('zs', 'umcot')],
}
OUTDIR = os.path.join(DATA_DIR, 'smoke_results' if SMOKE_ITEMS else RERUN_DIR)

class LogTee(io.TextIOBase):
    """Write everything to a log file; show status lines and every 50th progress line in the notebook."""
    def __init__(self, path):
        self.f, self.show = open(path, 'a'), sys.stdout
    def write(self, s):
        self.f.write(s); self.f.flush()
        for line in s.splitlines(True):
            head = line.split(' ', 1)[0]
            if not head.isdigit() or int(head) % 50 == 0:
                self.show.write(line)
        return len(s)

if MODE == 'run':
    os.makedirs(OUTDIR, exist_ok=True)
    for m in RUN_MODELS:
        since = dt.datetime.now(dt.timezone.utc).strftime('%Y-%m-%dT%H:%M')   # cost cap counts this run only
        for log, a in CMDS[m]:
            argv = a + COMMON + ['--outdir', OUTDIR, '--spend-this-model-only', '--spend-since', since]
            if SMOKE_ITEMS:
                argv += ['--limit', str(SMOKE_ITEMS)]
            logpath = os.path.join(OUTDIR, f'{log}.log')
            print(f'\n=== {m} ({log}); full log: {logpath}; "!!" lines are hard stops (402, daily quota, cost cap)')
            tee = LogTee(logpath)
            with contextlib.redirect_stdout(tee):
                run_eval_main(argv)
else:
    print('score_only: no API calls')

## Scores

**Step 13: Show the results table.** In run mode it scores the new outputs. In score-only mode it shows the combined table for all six models that the check above regenerated.

*Input:* scored results. *Output:* the summary table, shown below.

In [ ]:
# Score a re-run (score_main per model + combined summary); in score_only mode the tables were regenerated above.
from IPython.display import Markdown, display
LABELS = {'gpt-6-luna': 'GPT-6 Luna', 'gemma-4-26b-a4b-it': 'Gemma 4 26B-A4B', 'gemini-3.5-flash-lite': 'Gemini 3.5 Flash-Lite'}
if MODE == 'run':
    SCORE_DIR = OUTDIR
    done = [m for m in RUN_MODELS if os.path.isdir(os.path.join(SCORE_DIR, m))]
    with contextlib.redirect_stdout(io.StringIO()):
        for m in done:
            score_main(['--model', m, '--outdir', SCORE_DIR, '--ids', os.path.join(DATA_DIR, 'data/sample366_ids.csv')])
        score_extra_main(['--outdir', SCORE_DIR, '--models', *done, '--labels', *[LABELS[m] for m in done], '--paper-rows', 'api',
                          '--summary-name', 'summary_api_models_sample366.md', '--title', 'API models (re-run)'])
    SUMMARY = 'summary_api_models_sample366.md'
else:
    SCORE_DIR = os.path.join(DATA_DIR, 'results')
    SUMMARY = 'summary_all_models_sample366.md'   # score_only: all six models, regenerated by reproduce_main above
display(Markdown(open(os.path.join(SCORE_DIR, SUMMARY)).read()))

**Step 14: Per-model details.** Shows each API model's detailed score file: source-weighted accuracy, accuracy per source dataset, and reply script.

*Input:* score files. *Output:* tables shown below.

In [ ]:
# Per-model detail: source-weighted accuracy, per-dataset accuracy, reply script of reasoning outputs
for m in LABELS:
    f = os.path.join(SCORE_DIR, m, 'scores_sample366.md')
    if os.path.exists(f):
        display(Markdown(open(f).read()))

**Step 15: Package the results.** Zips the scored folder (raw outputs plus score files) so it can be downloaded or sent back.

*Input:* the scored folder. *Output:* `nlp_project_api_models_<folder>.zip` in the data folder.

In [ ]:
# ---- zip the scored folder (raw outputs + score files) for sending back ----
import shutil
z = shutil.make_archive(os.path.join(DATA_DIR, 'nlp_project_api_models_' + os.path.basename(SCORE_DIR)), 'zip',
                        root_dir=os.path.dirname(SCORE_DIR), base_dir=os.path.basename(SCORE_DIR))
print('wrote', z)
if IN_COLAB and MODE == 'run':
    from google.colab import files; files.download(z)